In [ ]:
import os
import getpass
import secrets
from secretsharing import PlaintextToHexSecretSharer
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend


def get_or_create_rsa_keys():
    private_path = "private_key.pem"
    public_path = "public_key.pem"

    if not os.path.exists(private_path):
        print("\n🔑 Khởi tạo hệ thống khóa RSA lần đầu...")
        password = getpass.getpass("Thiết lập mật khẩu bảo vệ file khóa RSA: ")
        
        private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
        
        # Lưu Private Key
        with open(private_path, "wb") as f:
            f.write(private_key.private_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PrivateFormat.PKCS8,
                encryption_algorithm=serialization.BestAvailableEncryption(password.encode())
            ))
        
        # Lưu Public Key
        public_key = private_key.public_key()
        with open(public_path, "wb") as f:
            f.write(public_key.public_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PublicFormat.SubjectPublicKeyInfo
            ))
        return private_key, public_key
    else:
        print ("LƯU Ý! KHÔNG CHIA SẺ MẬT KHẨU NÀY VỚI AI KHÁC! MẬT KHẨU NÀY CẦN THIẾT ĐỂ TRUY CẬP VÀ SỬ DỤNG KHÓA RSA CỦA BẠN.")
        password = getpass.getpass("\n🔑 Nhập mật khẩu: ")
        with open(private_path, "rb") as f:
            private_key = serialization.load_pem_private_key(f.read(), password=password.encode())
        with open(public_path, "rb") as f:
            public_key = serialization.load_pem_public_key(f.read())
        return private_key, public_key

# --- MÃ HÓA FILE VỚI AES ---

def encrypt_file(file_path):
    aes_key = os.urandom(32)  # Khóa 256-bit
    iv = os.urandom(16)
    
    with open(file_path, 'rb') as f:
        data = f.read()

    cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    encrypted_data = encryptor.update(data) + encryptor.finalize()

    with open(file_path + ".enc", 'wb') as f:
        f.write(iv + encrypted_data)
    
    return aes_key

# --- CHƯƠNG TRÌNH CHÍNH ---

# 1. Khởi tạo khóa RSA
try:
    priv_key, pub_key = get_or_create_rsa_keys()
except Exception as e:
    print(f"❌ Lỗi truy cập khóa: {e}")
    exit()

# 2. Mã hóa file và tạo Session Secret
target_file = input("\nNhập đường dẫn file cần mã hóa: ")
if os.path.exists(target_file):
    raw_aes_key = encrypt_file(target_file)
    session_salt = secrets.token_hex(4)
    # Kết hợp Khóa AES + Salt để tạo bí mật cần chia sẻ
    combined_secret = raw_aes_key.hex() + ":" + session_salt
    
    # Mã hóa bí mật này bằng RSA Public Key
    encrypted_session_secret = pub_key.encrypt(
        combined_secret.encode(),
        padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
    )
    print(f"✅ Đã mã hóa file. Session Salt của bạn là: {session_salt}")
else:
    print("File không tồn tại!")
    exit()

# 3. Chia sẻ bí mật (SSS)
n = int(input("\nTổng số thành viên phân phát mảnh: "))
k = round(n * 3 / 5)
shares = PlaintextToHexSecretSharer.split_secret(encrypted_session_secret.hex(), k, n)

print("\n--- CÁC MẢNH BÍ MẬT (HÃY CHIA CHO CÁC THÀNH VIÊN) ---")
for i, s in enumerate(shares, 1):
    print(f"Mảnh {i}: {s}")

# 4. Khôi phục và Giải mã
print(f"\n--- QUY TRÌNH KHÔI PHỤC (Cần {k} mảnh) ---")
subset = [input(f"Nhập mảnh thứ {i+1}: ") for i in range(k)]

try:
    recovered_hex = PlaintextToHexSecretSharer.recover_secret(subset)
    # Giải mã bằng RSA Private Key
    decrypted_combined = priv_key.decrypt(
        bytes.fromhex(recovered_hex),
        padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()), algorithm=hashes.SHA256(), label=None)
    ).decode()
    
    recovered_aes_key_hex, recovered_salt = decrypted_combined.split(":")
    
    print(f"\n🔓 Khôi phục thành công!")
    print(f"Khóa AES: {recovered_aes_key_hex}")
    print(f"Salt xác nhận: {recovered_salt}")

except Exception as e:
    print(f"❌ Khôi phục thất bại. Mảnh sai hoặc mật khẩu RSA sai! ({e})")



In [ ]:
import os, secrets
from flask import Flask, request, send_file, render_template_string
from secretsharing import PlaintextToHexSecretSharer
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend

app = Flask(__name__)

# --- RSA Key Management ---
def get_or_create_rsa_keys(password: str):
    private_path = "private_key.pem"
    public_path = "public_key.pem"

    if not os.path.exists(private_path):
        private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
        # Save private key (encrypted with password)
        with open(private_path, "wb") as f:
            f.write(private_key.private_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PrivateFormat.PKCS8,
                encryption_algorithm=serialization.BestAvailableEncryption(password.encode())
            ))
        # Save public key
        public_key = private_key.public_key()
        with open(public_path, "wb") as f:
            f.write(public_key.public_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PublicFormat.SubjectPublicKeyInfo
            ))
        return private_key, public_key
    else:
        with open(private_path, "rb") as f:
            private_key = serialization.load_pem_private_key(f.read(), password=password.encode())
        with open(public_path, "rb") as f:
            public_key = serialization.load_pem_public_key(f.read())
        return private_key, public_key

# --- AES File Encryption ---
def encrypt_file(file_path):
    aes_key = os.urandom(32)  # 256-bit key
    iv = os.urandom(16)
    with open(file_path, 'rb') as f:
        data = f.read()
    cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    encrypted_data = encryptor.update(data) + encryptor.finalize()
    enc_path = file_path + ".enc"
    with open(enc_path, 'wb') as f:
        f.write(iv + encrypted_data)
    return aes_key, enc_path

# --- Flask Routes ---
@app.route('/')
def index():
    return render_template_string('''
    <h2>Upload file để mã hóa</h2>
    <form method="post" action="/upload" enctype="multipart/form-data">
        <input type="file" name="file"><br><br>
        <input type="password" name="rsa_pass" placeholder="Mật khẩu RSA"><br><br>
        <input type="submit" value="Mã hóa">
    </form>
    ''')

@app.route('/upload', methods=['POST'])
def upload_file():
    uploaded_file = request.files['file']
    rsa_pass = request.form['rsa_pass']
    if uploaded_file.filename != '':
        os.makedirs("uploads", exist_ok=True)
        file_path = os.path.join("uploads", uploaded_file.filename)
        uploaded_file.save(file_path)

        # RSA keys
        priv_key, pub_key = get_or_create_rsa_keys(rsa_pass)

        # Encrypt file with AES
        raw_aes_key, enc_path = encrypt_file(file_path)
        session_salt = secrets.token_hex(4)
        combined_secret = raw_aes_key.hex() + ":" + session_salt


In [ ]:
import os, secrets
from flask import Flask, request, send_file, render_template_string
from secretsharing import PlaintextToHexSecretSharer
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.primitives.asymmetric import rsa, padding
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.backends import default_backend

app = Flask(__name__)

# --- RSA Key Management ---
def get_or_create_rsa_keys(password: str):
    private_path = "private_key.pem"
    public_path = "public_key.pem"

    if not os.path.exists(private_path):
        private_key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
        # Save private key (encrypted with password)
        with open(private_path, "wb") as f:
            f.write(private_key.private_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PrivateFormat.PKCS8,
                encryption_algorithm=serialization.BestAvailableEncryption(password.encode())
            ))
        # Save public key
        public_key = private_key.public_key()
        with open(public_path, "wb") as f:
            f.write(public_key.public_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PublicFormat.SubjectPublicKeyInfo
            ))
        return private_key, public_key
    else:
        with open(private_path, "rb") as f:
            private_key = serialization.load_pem_private_key(f.read(), password=password.encode())
        with open(public_path, "rb") as f:
            public_key = serialization.load_pem_public_key(f.read())
        return private_key, public_key

# --- AES File Encryption ---
def encrypt_file(file_path):
    aes_key = os.urandom(32)  # 256-bit key
    iv = os.urandom(16)
    with open(file_path, 'rb') as f:
        data = f.read()
    cipher = Cipher(algorithms.AES(aes_key), modes.CFB(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    encrypted_data = encryptor.update(data) + encryptor.finalize()
    enc_path = file_path + ".enc"
    with open(enc_path, 'wb') as f:
        f.write(iv + encrypted_data)
    return aes_key, enc_path

# --- Flask Routes ---
@app.route('/')
def index():
    return render_template_string('''
    <h2>Upload file để mã hóa</h2>
    <form method="post" action="/upload" enctype="multipart/form-data">
        <input type="file" name="file"><br><br>
        <input type="password" name="rsa_pass" placeholder="Mật khẩu RSA"><br><br>
        <input type="submit" value="Mã hóa">
    </form>
    ''')

@app.route('/upload', methods=['POST'])
def upload_file():
    uploaded_file = request.files['file']
    rsa_pass = request.form['rsa_pass']
    if uploaded_file.filename != '':
        os.makedirs("uploads", exist_ok=True)
        file_path = os.path.join("uploads", uploaded_file.filename)
        uploaded_file.save(file_path)

        # RSA keys
        priv_key, pub_key = get_or_create_rsa_keys(rsa_pass)

        # Encrypt file with AES
        raw_aes_key, enc_path = encrypt_file(file_path)
        session_salt = secrets.token_hex(4)
        combined_secret = raw_aes_key.hex() + ":" + session_salt

        # Encrypt combined secret with RSA
        encrypted_session_secret = pub_key.encrypt(
            combined_secret.encode(),
            padding.OAEP(mgf=padding.MGF1(algorithm=hashes.SHA256()),
                         algorithm=hashes.SHA256(), label=None)
        )

        # Split secret into shares
        n, k = 5, 3  # ví dụ: 5 thành viên, cần 3 mảnh
        shares = PlaintextToHexSecretSharer.split_secret(encrypted_session_secret.hex(), k, n)

        # Hiển thị shares + cho tải file .enc
        html = "<h3>File đã mã hóa xong!</h3>"
        html += f"<p>Salt phiên: {session_salt}</p>"
        html += "<p>Các mảnh bí mật:</p><ul>"
        for i, s in enumerate(shares, 1):
            html += f"<li>Mảnh {i}: {s}</li>"
        html += "</ul>"
        html += f'<a href="/download/{os.path.basename(enc_path)}">Tải file mã hóa</a>'
        return html
    return "Không có file nào được tải lên!"

@app.route('/download/<filename>')
def download_file(filename):
    return send_file(os.path.join("uploads", filename), as_attachment=True)

if __name__ == '__main__':
    app.run(debug=True)